# Results dashboard
Reads only newly generated v3 reports. It shows available individual runs and, after all 15 jobs, the frozen campaign aggregate.


In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Markdown, Image
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
RUN_ID='final_3x5_v1'
run_reports=sorted((ROOT/'reports'/RUN_ID/'runs').glob('seed_*_fold_*/report.json')) if (ROOT/'reports'/RUN_ID/'runs').exists() else []
print(f'Completed reports: {len(run_reports)}/15')


In [ ]:
rows=[]
for path in run_reports:
    r=json.loads(path.read_text())
    p=r['results']['phytocore']['metrics']; vlm=r['results']['vlm']; v=vlm['metrics']; layers=vlm.get('generation_layers',{})
    constrained=layers.get('constrained_top1',{}); reranked=layers.get('verifier_reranked_nbest',{})
    rows.append({'seed':r['seed'],'fold':r['fold'],'GrainWeight MAE':p['regression']['GrainWeight']['mae'],'Gsw MAE':p['regression']['Gsw']['mae'],'PhiPS2 MAE':p['regression']['PhiPS2']['mae'],'Fertilizer accuracy':p['classification']['Fertilizer']['accuracy'],'Cultivar accuracy':p['classification']['Cultivar']['accuracy'],'Raw VLM acceptance':v['raw_reference_acceptance'],'Raw class agreement':v['raw_class_agreement_with_phytocore'],'Summary acceptance':v['summary_acceptance'],'Constrained acceptance':constrained.get('raw_reference_acceptance'),'N-best verified acceptance':reranked.get('raw_reference_acceptance'),'Verified consistency':v['verified_evidence_consistency']})
display(pd.DataFrame(rows))


In [ ]:
aggregate=ROOT/'reports'/RUN_ID/'campaign'/'aggregate_report.json'
if aggregate.exists():
    report=json.loads(aggregate.read_text())
    display(pd.DataFrame(report['metrics_across_runs']).T)
    figure=ROOT/'reports'/RUN_ID/'campaign'/'figures'/'campaign_metrics.png'
    if figure.exists(): display(Image(filename=str(figure)))
else:
    display(Markdown('Campaign aggregate is not available until all 15 runs finish.'))
